In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Домашнее задание HW12\n",
    "## Временные ряды: корректная валидация, лаговые признаки, GRU\n",
    "\n",
    "Выполнил: [Ваше имя]\n",
    "Дата: 2025-03-27"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Импорты, seed, устройство"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.linear_model import Ridge\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.metrics import mean_absolute_error, mean_squared_error\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "from torch.utils.data import Dataset, DataLoader\n",
    "import json\n",
    "import os\n",
    "import random\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Фиксация seed для воспроизводимости\n",
    "SEED = 42\n",
    "random.seed(SEED)\n",
    "np.random.seed(SEED)\n",
    "torch.manual_seed(SEED)\n",
    "if torch.cuda.is_available():\n",
    "    torch.cuda.manual_seed(SEED)\n",
    "\n",
    "# Определяем устройство\n",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
    "print(f\"Using device: {device}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Загрузка и первичный анализ данных"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Загрузка\n",
    "df = pd.read_csv('S12-hw-dataset.csv')\n",
    "print(f\"Исходный размер: {df.shape}\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Преобразование даты и сортировка\n",
    "df['date'] = pd.to_datetime(df['date'])\n",
    "df = df.sort_values('date').reset_index(drop=True)\n",
    "\n",
    "# Проверка пропусков\n",
    "print(\"Пропуски:\")\n",
    "print(df.isnull().sum())\n",
    "\n",
    "# Если есть пропуски, заполним ближайшим предыдущим значением (ffill)\n",
    "if df.isnull().any().any():\n",
    "    df = df.fillna(method='ffill')\n",
    "    print(\"Пропуски заполнены методом ffill\")\n",
    "\n",
    "# Диапазон дат\n",
    "print(f\"Диапазон дат: {df['date'].min()} - {df['date'].max()}\")\n",
    "print(f\"Количество наблюдений: {len(df)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# График временного ряда\n",
    "plt.figure(figsize=(12,5))\n",
    "plt.plot(df['date'], df['target'], label='target')\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Target')\n",
    "plt.title('Временной ряд target')\n",
    "plt.legend()\n",
    "plt.grid(True)\n",
    "plt.show()\n",
    "\n",
    "# Комментарий: оценим тренд, сезонность, выбросы (визуально)\n",
    "# Здесь можно добавить текстовый комментарий в markdown ячейке ниже."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "**Комментарий к ряду:**\n",
    "- На графике виден [тренд/сезонность/выбросы].\n",
    "- Ряд выглядит [стационарным/нестационарным].\n",
    "- Для дальнейшего анализа используем лаговые и rolling-признаки, которые могут уловить сезонность."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Временное разбиение (temporal split)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Определим пропорции: 70% train, 15% val, 15% test\n",
    "n = len(df)\n",
    "train_end = int(0.7 * n)\n",
    "val_end = train_end + int(0.15 * n)\n",
    "\n",
    "train_df = df.iloc[:train_end]\n",
    "val_df = df.iloc[train_end:val_end]\n",
    "test_df = df.iloc[val_end:]\n",
    "\n",
    "print(f\"Train: {len(train_df)} ({train_df['date'].min()} - {train_df['date'].max()})\")\n",
    "print(f\"Val: {len(val_df)} ({val_df['date'].min()} - {val_df['date'].max()})\")\n",
    "print(f\"Test: {len(test_df)} ({test_df['date'].min()} - {test_df['date'].max()})\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Визуализация разбиения\n",
    "plt.figure(figsize=(12,5))\n",
    "plt.plot(train_df['date'], train_df['target'], label='Train', color='blue')\n",
    "plt.plot(val_df['date'], val_df['target'], label='Validation', color='orange')\n",
    "plt.plot(test_df['date'], test_df['target'], label='Test', color='green')\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Target')\n",
    "plt.title('Temporal split: Train / Validation / Test')\n",
    "plt.legend()\n",
    "plt.grid(True)\n",
    "plt.savefig('artifacts/figures/series_split.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "# Пояснение\n",
    "print(\"\\nПочему random split некорректен: при случайном перемешивании нарушается хронология, \"\n",
    "      \"в обучающую выборку могут попасть будущие значения относительно валидации, что приводит \"\n",
    "      \"к утечке информации и нереалистично оптимистичной оценке качества.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Построение признаков для baseline-моделей\n",
    "\n",
    "Создадим лаговые, rolling и календарные признаки. Важно: признаки строятся на основе только train + возможно валидация без использования будущих значений. При прогнозировании будем использовать признаки, доступные на момент прогноза."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def create_features(df_orig, window=7):\n",
    "    \"\"\"\n",
    "    Добавляет признаки к копии датафрейма.\n",
    "    Предполагается, что df_orig уже отсортирован по времени.\n",
    "    \"\"\"\n",
    "    df = df_orig.copy()\n",
    "    # Лаги\n",
    "    df['lag_1'] = df['target'].shift(1)\n",
    "    df['lag_7'] = df['target'].shift(7)\n",
    "    df['lag_14'] = df['target'].shift(14)\n",
    "    # Rolling статистики (с использованием shift, чтобы избежать утечки)\n",
    "    df['rolling_mean_7'] = df['target'].shift(1).rolling(window).mean()\n",
    "    df['rolling_std_7'] = df['target'].shift(1).rolling(window).std()\n",
    "    # Календарные\n",
    "    df['day_of_week'] = df['date'].dt.dayofweek\n",
    "    # Для простоты можно добавить месяц, квартал и т.п.\n",
    "    df['month'] = df['date'].dt.month\n",
    "    return df\n",
    "\n",
    "# Применим к каждой части отдельно, чтобы избежать утечки через shift при склейке\n",
    "train_feat = create_features(train_df)\n",
    "val_feat = create_features(val_df)\n",
    "test_feat = create_features(test_df)\n",
    "\n",
    "# Удалим строки с NaN из-за shift (первые строки каждой части)\n",
    "train_feat = train_feat.dropna().reset_index(drop=True)\n",
    "val_feat = val_feat.dropna().reset_index(drop=True)\n",
    "test_feat = test_feat.dropna().reset_index(drop=True)\n",
    "\n",
    "print(f\"Train после удаления NaN: {len(train_feat)}\")\n",
    "print(f\"Val после удаления NaN: {len(val_feat)}\")\n",
    "print(f\"Test после удаления NaN: {len(test_feat)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Эксперименты с baseline-моделями\n",
    "\n",
    "### 5.1. Наивный прогноз (B1) – последнее значение"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def naive_last(y_true, y_pred_shifted):\n",
    "    \"\"\"Прогноз = последнее известное значение. Здесь y_pred_shifted - сдвиг на 1.\"\"\"\n",
    "    return y_true.shift(1).fillna(method='bfill')\n",
    "\n",
    "# На валидации\n",
    "y_val = val_feat['target']\n",
    "y_val_naive = val_feat['lag_1']  # последнее наблюдение\n",
    "\n",
    "mae_val_b1 = mean_absolute_error(y_val, y_val_naive)\n",
    "rmse_val_b1 = np.sqrt(mean_squared_error(y_val, y_val_naive))\n",
    "mape_val_b1 = np.mean(np.abs((y_val - y_val_naive) / y_val)) * 100\n",
    "\n",
    "print(f\"B1 (naive-last) на валидации: MAE={mae_val_b1:.4f}, RMSE={rmse_val_b1:.4f}, MAPE={mape_val_b1:.2f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 5.2. Скользящее среднее (B2) – окно 7"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Используем rolling_mean_7 как прогноз\n",
    "y_val_ma = val_feat['rolling_mean_7']\n",
    "\n",
    "mae_val_b2 = mean_absolute_error(y_val, y_val_ma)\n",
    "rmse_val_b2 = np.sqrt(mean_squared_error(y_val, y_val_ma))\n",
    "mape_val_b2 = np.mean(np.abs((y_val - y_val_ma) / y_val)) * 100\n",
    "\n",
    "print(f\"B2 (moving-average) на валидации: MAE={mae_val_b2:.4f}, RMSE={rmse_val_b2:.4f}, MAPE={mape_val_b2:.2f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 5.3. Ridge на признаках (B3)\n",
    "\n",
    "Признаки: лаги, rolling, календарные. Масштабируем."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Выбираем признаки (исключаем target, date)\n",
    "feature_cols = ['lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_std_7', 'day_of_week', 'month']\n",
    "\n",
    "# Масштабирование: fit на train\n",
    "scaler_X = StandardScaler()\n",
    "scaler_y = StandardScaler()\n",
    "\n",
    "X_train = scaler_X.fit_transform(train_feat[feature_cols])\n",
    "y_train = scaler_y.fit_transform(train_feat[['target']]).ravel()\n",
    "\n",
    "X_val = scaler_X.transform(val_feat[feature_cols])\n",
    "y_val_scaled = scaler_y.transform(val_feat[['target']]).ravel()\n",
    "\n",
    "# Ridge модель\n",
    "ridge = Ridge(alpha=1.0, random_state=SEED)\n",
    "ridge.fit(X_train, y_train)\n",
    "\n",
    "# Прогноз на валидации\n",
    "y_pred_val_scaled = ridge.predict(X_val)\n",
    "y_pred_val = scaler_y.inverse_transform(y_pred_val_scaled.reshape(-1,1)).ravel()\n",
    "\n",
    "mae_val_b3 = mean_absolute_error(val_feat['target'], y_pred_val)\n",
    "rmse_val_b3 = np.sqrt(mean_squared_error(val_feat['target'], y_pred_val))\n",
    "mape_val_b3 = np.mean(np.abs((val_feat['target'] - y_pred_val) / val_feat['target'])) * 100\n",
    "\n",
    "print(f\"B3 (Ridge) на валидации: MAE={mae_val_b3:.4f}, RMSE={rmse_val_b3:.4f}, MAPE={mape_val_b3:.2f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Подготовка данных для GRU\n",
    "\n",
    "Создадим последовательности фиксированной длины `window_size`. Будем прогнозировать следующее значение (one-step ahead)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "WINDOW_SIZE = 14  # гиперпараметр, можно подбирать\n",
    "\n",
    "class TimeSeriesDataset(Dataset):\n",
    "    def __init__(self, series, window_size):\n",
    "        self.series = series\n",
    "        self.window_size = window_size\n",
    "        self.data = []\n",
    "        self.targets = []\n",
    "        for i in range(len(series) - window_size):\n",
    "            self.data.append(series[i:i+window_size])\n",
    "            self.targets.append(series[i+window_size])\n",
    "        self.data = np.array(self.data, dtype=np.float32)\n",
    "        self.targets = np.array(self.targets, dtype=np.float32)\n",
    "    \n",
    "    def __len__(self):\n",
    "        return len(self.data)\n",
    "    \n",
    "    def __getitem__(self, idx):\n",
    "        return torch.tensor(self.data[idx]), torch.tensor(self.targets[idx])\n",
    "\n",
    "# Для GRU используем только target, масштабируем\n",
    "scaler_gru = StandardScaler()\n",
    "train_series_scaled = scaler_gru.fit_transform(train_df[['target']]).ravel()\n",
    "val_series_scaled = scaler_gru.transform(val_df[['target']]).ravel()\n",
    "test_series_scaled = scaler_gru.transform(test_df[['target']]).ravel()\n",
    "\n",
    "# Создаём датасеты\n",
    "train_dataset = TimeSeriesDataset(train_series_scaled, WINDOW_SIZE)\n",
    "val_dataset = TimeSeriesDataset(val_series_scaled, WINDOW_SIZE)\n",
    "test_dataset = TimeSeriesDataset(test_series_scaled, WINDOW_SIZE)\n",
    "\n",
    "BATCH_SIZE = 32\n",
    "train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)\n",
    "val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)\n",
    "test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)\n",
    "\n",
    "print(f\"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Определение модели GRU"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class GRUModel(nn.Module):\n",
    "    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1, dropout=0.0):\n",
    "        super().__init__()\n",
    "        self.hidden_size = hidden_size\n",
    "        self.num_layers = num_layers\n",
    "        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)\n",
    "        self.fc = nn.Linear(hidden_size, output_size)\n",
    "    \n",
    "    def forward(self, x):\n",
    "        # x shape: (batch, seq_len, input_size)\n",
    "        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)\n",
    "        out, _ = self.gru(x, h0)\n",
    "        out = self.fc(out[:, -1, :])  # последний выход\n",
    "        return out\n",
    "\n",
    "model = GRUModel(input_size=1, hidden_size=64, num_layers=2).to(device)\n",
    "print(model)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Функции обучения и валидации"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def train_epoch(model, loader, criterion, optimizer, device):\n",
    "    model.train()\n",
    "    total_loss = 0\n",
    "    for X_batch, y_batch in loader:\n",
    "        X_batch = X_batch.unsqueeze(-1).to(device)  # добавить размерность input_size=1\n",
    "        y_batch = y_batch.to(device).unsqueeze(1)\n",
    "        optimizer.zero_grad()\n",
    "        outputs = model(X_batch)\n",
    "        loss = criterion(outputs, y_batch)\n",
    "        loss.backward()\n",
    "        optimizer.step()\n",
    "        total_loss += loss.item() * X_batch.size(0)\n",
    "    return total_loss / len(loader.dataset)\n",
    "\n",
    "def validate(model, loader, criterion, device, scaler=None):\n",
    "    model.eval()\n",
    "    total_loss = 0\n",
    "    all_preds = []\n",
    "    all_targets = []\n",
    "    with torch.no_grad():\n",
    "        for X_batch, y_batch in loader:\n",
    "            X_batch = X_batch.unsqueeze(-1).to(device)\n",
    "            y_batch = y_batch.to(device).unsqueeze(1)\n",
    "            outputs = model(X_batch)\n",
    "            loss = criterion(outputs, y_batch)\n",
    "            total_loss += loss.item() * X_batch.size(0)\n",
    "            all_preds.append(outputs.cpu().numpy())\n",
    "            all_targets.append(y_batch.cpu().numpy())\n",
    "    all_preds = np.concatenate(all_preds, axis=0)\n",
    "    all_targets = np.concatenate(all_targets, axis=0)\n",
    "    if scaler:\n",
    "        all_preds = scaler.inverse_transform(all_preds)\n",
    "        all_targets = scaler.inverse_transform(all_targets)\n",
    "    mae = mean_absolute_error(all_targets, all_preds)\n",
    "    rmse = np.sqrt(mean_squared_error(all_targets, all_preds))\n",
    "    mape = np.mean(np.abs((all_targets - all_preds) / all_targets)) * 100\n",
    "    return total_loss / len(loader.dataset), mae, rmse, mape\n",
    "\n",
    "criterion = nn.MSELoss()\n",
    "optimizer = torch.optim.Adam(model.parameters(), lr=0.001)\n",
    "num_epochs = 50\n",
    "\n",
    "# Сохранение истории\n",
    "train_losses = []\n",
    "val_losses = []\n",
    "best_val_mae = float('inf')\n",
    "best_model_state = None\n",
    "\n",
    "for epoch in range(num_epochs):\n",
    "    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)\n",
    "    val_loss, val_mae, val_rmse, val_mape = validate(model, val_loader, criterion, device, scaler=scaler_gru)\n",
    "    train_losses.append(train_loss)\n",
    "    val_losses.append(val_loss)\n",
    "    \n",
    "    if val_mae < best_val_mae:\n",
    "        best_val_mae = val_mae\n",
    "        best_model_state = model.state_dict().copy()\n",
    "        torch.save(best_model_state, 'artifacts/best_gru.pt')\n",
    "        # Сохраним конфиг\n",
    "        config = {\n",
    "            'window_size': WINDOW_SIZE,\n",
    "            'hidden_size': 64,\n",
    "            'num_layers': 2,\n",
    "            'batch_size': BATCH_SIZE,\n",
    "            'learning_rate': 0.001,\n",
    "            'seed': SEED,\n",
    "            'scaler_mean': scaler_gru.mean_[0],\n",
    "            'scaler_scale': scaler_gru.scale_[0]\n",
    "        }\n",
    "        with open('artifacts/best_gru_config.json', 'w') as f:\n",
    "            json.dump(config, f, indent=4)\n",
    "    \n",
    "    if (epoch+1) % 10 == 0:\n",
    "        print(f\"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Val MAE: {val_mae:.4f}\")\n",
    "\n",
    "# После обучения загрузим лучшую модель\n",
    "model.load_state_dict(torch.load('artifacts/best_gru.pt'))\n",
    "print(f\"Лучшая модель сохранена с Val MAE = {best_val_mae:.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Кривые обучения\n",
    "plt.figure(figsize=(10,5))\n",
    "plt.plot(range(1, num_epochs+1), train_losses, label='Train Loss')\n",
    "plt.plot(range(1, num_epochs+1), val_losses, label='Val Loss')\n",
    "plt.xlabel('Epoch')\n",
    "plt.ylabel('MSE Loss')\n",
    "plt.title('Learning Curves (GRU)')\n",
    "plt.legend()\n",
    "plt.grid(True)\n",
    "plt.savefig('artifacts/figures/gru_learning_curves.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Финальная оценка на тесте для лучшей модели"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Определим лучшую модель по валидации: выберем между B1, B2, B3, R1\n",
    "models_val = {\n",
    "    'B1': mae_val_b1,\n",
    "    'B2': mae_val_b2,\n",
    "    'B3': mae_val_b3,\n",
    "    'R1': best_val_mae\n",
    "}\n",
    "best_model_name = min(models_val, key=models_val.get)\n",
    "print(f\"Лучшая модель по Val MAE: {best_model_name} с MAE = {models_val[best_model_name]:.4f}\")\n",
    "\n",
    "# Вычисляем тестовые метрики для лучшей модели\n",
    "if best_model_name == 'B1':\n",
    "    y_test = test_feat['target']\n",
    "    y_pred = test_feat['lag_1']\n",
    "    mae_test = mean_absolute_error(y_test, y_pred)\n",
    "    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))\n",
    "    mape_test = np.mean(np.abs((y_test - y_pred) / y_test)) * 100\n",
    "elif best_model_name == 'B2':\n",
    "    y_test = test_feat['target']\n",
    "    y_pred = test_feat['rolling_mean_7']\n",
    "    mae_test = mean_absolute_error(y_test, y_pred)\n",
    "    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))\n",
    "    mape_test = np.mean(np.abs((y_test - y_pred) / y_test)) * 100\n",
    "elif best_model_name == 'B3':\n",
    "    X_test = scaler_X.transform(test_feat[feature_cols])\n",
    "    y_pred_scaled = ridge.predict(X_test)\n",
    "    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1,1)).ravel()\n",
    "    y_test = test_feat['target']\n",
    "    mae_test = mean_absolute_error(y_test, y_pred)\n",
    "    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))\n",
    "    mape_test = np.mean(np.abs((y_test - y_pred) / y_test)) * 100\n",
    "else:  # R1\n",
    "    # Прогноз на тесте\n",
    "    model.eval()\n",
    "    all_preds = []\n",
    "    all_targets = []\n",
    "    with torch.no_grad():\n",
    "        for X_batch, y_batch in test_loader:\n",
    "            X_batch = X_batch.unsqueeze(-1).to(device)\n",
    "            outputs = model(X_batch)\n",
    "            all_preds.append(outputs.cpu().numpy())\n",
    "            all_targets.append(y_batch.numpy())\n",
    "    all_preds = np.concatenate(all_preds, axis=0)\n",
    "    all_targets = np.concatenate(all_targets, axis=0)\n",
    "    # Обратное масштабирование\n",
    "    all_preds = scaler_gru.inverse_transform(all_preds)\n",
    "    all_targets = scaler_gru.inverse_transform(all_targets)\n",
    "    mae_test = mean_absolute_error(all_targets, all_preds)\n",
    "    rmse_test = np.sqrt(mean_squared_error(all_targets, all_preds))\n",
    "    mape_test = np.mean(np.abs((all_targets - all_preds) / all_targets)) * 100\n",
    "    # Для графика возьмём последние, чтобы показать\n",
    "    y_test = all_targets\n",
    "    y_pred = all_preds\n",
    "\n",
    "print(f\"Тестовая оценка лучшей модели ({best_model_name}): MAE={mae_test:.4f}, RMSE={rmse_test:.4f}, MAPE={mape_test:.2f}%\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# График факта и прогноза на тесте (если модель R1, иначе B3)\n",
    "if best_model_name == 'R1':\n",
    "    plt.figure(figsize=(12,5))\n",
    "    # Используем только тестовую часть\n",
    "    plt.plot(test_df['date'].iloc[WINDOW_SIZE:], y_test, label='Actual', color='blue')\n",
    "    plt.plot(test_df['date'].iloc[WINDOW_SIZE:], y_pred, label='Predicted (GRU)', color='red', linestyle='--')\n",
    "    plt.xlabel('Date')\n",
    "    plt.ylabel('Target')\n",
    "    plt.title('Best Model Forecast on Test Set')\n",
    "    plt.legend()\n",
    "    plt.grid(True)\n",
    "    plt.savefig('artifacts/figures/best_forecast_test.png', dpi=150, bbox_inches='tight')\n",
    "    plt.show()\n",
    "else:\n",
    "    # Для B3 тоже можно, но в задании требуется для лучшей\n",
    "    # Сделаем для R1, если лучшая, иначе для B3 (если лучшая)\n",
    "    if best_model_name == 'B3':\n",
    "        plt.figure(figsize=(12,5))\n",
    "        plt.plot(test_feat['date'], y_test, label='Actual', color='blue')\n",
    "        plt.plot(test_feat['date'], y_pred, label='Predicted (Ridge)', color='red', linestyle='--')\n",
    "        plt.xlabel('Date')\n",
    "        plt.ylabel('Target')\n",
    "        plt.title('Best Model Forecast on Test Set')\n",
    "        plt.legend()\n",
    "        plt.grid(True)\n",
    "        plt.savefig('artifacts/figures/best_forecast_test.png', dpi=150, bbox_inches='tight')\n",
    "        plt.show()\n",
    "    else:\n",
    "        print(\"Для B1/B2 график не строится, но можно было бы\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Формирование runs.csv"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Собираем все результаты в DataFrame\n",
    "runs = []\n",
    "\n",
    "# B1\n",
    "runs.append({\n",
    "    'experiment_id': 'B1',\n",
    "    'task': 'forecasting',\n",
    "    'dataset': 'S12-hw-dataset',\n",
    "    'seed': SEED,\n",
    "    'split_summary': f'train:{train_end} val:{val_end-train_end} test:{n-val_end}',\n",
    "    'window_size': None,\n",
    "    'horizon': 1,\n",
    "    'model_summary': 'naive-last',\n",
    "    'features_summary': 'none',\n",
    "    'scaler': None,\n",
    "    'optimizer': None,\n",
    "    'lr': None,\n",
    "    'epochs_trained': None,\n",
    "    'best_val_mae': mae_val_b1,\n",
    "    'best_val_rmse': rmse_val_b1,\n",
    "    'best_val_mape': mape_val_b1,\n",
    "    'test_mae': mae_test if best_model_name=='B1' else None,\n",
    "    'test_rmse': rmse_test if best_model_name=='B1' else None,\n",
    "    'test_mape': mape_test if best_model_name=='B1' else None,\n",
    "    'notes': ''\n",
    "})\n",
    "\n",
    "# B2\n",
    "runs.append({\n",
    "    'experiment_id': 'B2',\n",
    "    'task': 'forecasting',\n",
    "    'dataset': 'S12-hw-dataset',\n",
    "    'seed': SEED,\n",
    "    'split_summary': f'train:{train_end} val:{val_end-train_end} test:{n-val_end}',\n",
    "    'window_size': None,\n",
    "    'horizon': 1,\n",
    "    'model_summary': 'moving-average (window=7)',\n",
    "    'features_summary': 'none',\n",
    "    'scaler': None,\n",
    "    'optimizer': None,\n",
    "    'lr': None,\n",
    "    'epochs_trained': None,\n",
    "    'best_val_mae': mae_val_b2,\n",
    "    'best_val_rmse': rmse_val_b2,\n",
    "    'best_val_mape': mape_val_b2,\n",
    "    'test_mae': mae_test if best_model_name=='B2' else None,\n",
    "    'test_rmse': rmse_test if best_model_name=='B2' else None,\n",
    "    'test_mape': mape_test if best_model_name=='B2' else None,\n",
    "    'notes': ''\n",
    "})\n",
    "\n",
    "# B3\n",
    "runs.append({\n",
    "    'experiment_id': 'B3',\n",
    "    'task': 'forecasting',\n",
    "    'dataset': 'S12-hw-dataset',\n",
    "    'seed': SEED,\n",
    "    'split_summary': f'train:{train_end} val:{val_end-train_end} test:{n-val_end}',\n",
    "    'window_size': None,\n",
    "    'horizon': 1,\n",
    "    'model_summary': 'Ridge (alpha=1)',\n",
    "    'features_summary': ','.join(feature_cols),\n",
    "    'scaler': 'StandardScaler',\n",
    "    'optimizer': None,\n",
    "    'lr': None,\n",
    "    'epochs_trained': None,\n",
    "    'best_val_mae': mae_val_b3,\n",
    "    'best_val_rmse': rmse_val_b3,\n",
    "    'best_val_mape': mape_val_b3,\n",
    "    'test_mae': mae_test if best_model_name=='B3' else None,\n",
    "    'test_rmse': rmse_test if best_model_name=='B3' else None,\n",
    "    'test_mape': mape_test if best_model_name=='B3' else None,\n",
    "    'notes': ''\n",
    "})\n",
    "\n",
    "# R1\n",
    "runs.append({\n",
    "    'experiment_id': 'R1',\n",
    "    'task': 'forecasting',\n",
    "    'dataset': 'S12-hw-dataset',\n",
    "    'seed': SEED,\n",
    "    'split_summary': f'train:{train_end} val:{val_end-train_end} test:{n-val_end}',\n",
    "    'window_size': WINDOW_SIZE,\n",
    "    'horizon': 1,\n",
    "    'model_summary': 'GRU (hidden=64, layers=2)',\n",
    "    'features_summary': 'target only',\n",
    "    'scaler': 'StandardScaler',\n",
    "    'optimizer': 'Adam',\n",
    "    'lr': 0.001,\n",
    "    'epochs_trained': num_epochs,\n",
    "    'best_val_mae': best_val_mae,\n",
    "    'best_val_rmse': None,  # можно добавить, но для краткости\n",
    "    'best_val_mape': None,\n",
    "    'test_mae': mae_test if best_model_name=='R1' else None,\n",
    "    'test_rmse': rmse_test if best_model_name=='R1' else None,\n",
    "    'test_mape': mape_test if best_model_name=='R1' else None,\n",
    "    'notes': ''\n",
    "})\n",
    "\n",
    "runs_df = pd.DataFrame(runs)\n",
    "runs_df.to_csv('artifacts/runs.csv', index=False)\n",
    "print(\"runs.csv сохранён\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Сравнение baseline-моделей на валидации"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Построим столбчатую диаграмму MAE\n",
    "models = ['B1', 'B2', 'B3', 'R1']\n",
    "val_mae = [mae_val_b1, mae_val_b2, mae_val_b3, best_val_mae]\n",
    "plt.figure(figsize=(8,5))\n",
    "plt.bar(models, val_mae, color=['gray', 'lightblue', 'orange', 'green'])\n",
    "plt.ylabel('MAE')\n",
    "plt.title('Validation MAE Comparison')\n",
    "for i, v in enumerate(val_mae):\n",
    "    plt.text(i, v + 0.01, f'{v:.4f}', ha='center')\n",
    "plt.savefig('artifacts/figures/baselines_compare.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Заключение\n",
    "Все эксперименты выполнены, результаты зафиксированы. Лучшая модель по валидации – [имя], тестовая MAE = [значение].\n",
    "\n",
    "**Ключевые выводы:**\n",
    "- Temporal split позволяет избежать утечки данных.\n",
    "- Лаговые и rolling-признаки улучшают прогноз.\n",
    "- GRU показывает [сопоставимые/лучшие] результаты, но требует больше ресурсов.\n",
    "\n",
    "**Артефакты:**\n",
    "- runs.csv, best_gru.pt, best_gru_config.json, графики в папке figures."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}